#Introduction to GPU Acceleration
### 🔍 Why Use GPUs?

GPUs are optimized for large-scale parallel computation, making them ideal for matrix-heavy tasks in deep learning. In this lab, you'll compare training times and performance on CPU vs GPU and learn how to write GPU-efficient code.


##Check device availability

## TensorFlow

In [1]:
import tensorflow as tf
print("Is GPU available?", tf.config.list_physical_devices('GPU'))

Is GPU available? [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


##PyTorch

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## 🚀 Moving Models and Data to GPU

To fully utilize the GPU, both the model and input data must be moved to the GPU device. This ensures the computation is performed on the GPU instead of the CPU.

Let's see how to do this in both TensorFlow and PyTorch.


##TensorFlow - Using GPU Automatically

In [3]:
# TensorFlow uses GPU by default when available
import tensorflow as tf

with tf.device('/GPU:0'):  # or '/CPU:0' for CPU
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)
    print("Operation completed on:", c.device)


Operation completed on: /job:localhost/replica:0/task:0/device:GPU:0


##PyTorch - Manual GPU Transfer

In [4]:
import torch

# Use 'cuda' if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example tensor operation on GPU
a = torch.randn(1000, 1000).to(device)
b = torch.randn(1000, 1000).to(device)
c = torch.matmul(a, b)

print("Tensor 'c' is on device:", c.device)


Tensor 'c' is on device: cuda:0


##Measuring Training Time on CPU vs GPU

## ⏱️ Performance Benchmark: CPU vs GPU

We'll train a simple model on the MNIST dataset using both CPU and GPU. This will help us visualize the speedup provided by GPU acceleration.

Steps:
- Train on CPU and measure the time
- Train on GPU and measure the time
- Compare the difference


In [5]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)




##Train on CPU

In [6]:
# Train on CPU
def train_on_cpu():
    device = torch.device("cpu")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ CPU training time: {end_time - start_time:.2f} sec")

train_on_cpu()

✅ CPU training time: 9.30 sec


##Train on GPU
Change your runtine to T4 GPU and run the following code block

In [7]:
# Train on GPU
def train_on_gpu():
    if not torch.cuda.is_available():
        print("🚫 CUDA not available on this system.")
        return

    device = torch.device("cuda")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ GPU training time: {end_time - start_time:.2f} sec")

train_on_gpu()


✅ GPU training time: 8.31 sec


I suppose we got lesser time for the gpu, it makes more difference on larger models that have more number of layers and filters, gpu speeds up the matrix multiplication due to the presence of numerous small cores.

**So here's an activity for you**
##Use tensorflow to train a model on the MNIST digits dataset on both gpu and cpu and examine which one works faster.

Use your custom number of layers and filters to experiment with the hyperparameters of the model.

In [8]:
# ============================================
# Activity: TensorFlow CPU vs GPU Benchmark
# ============================================

import tensorflow as tf
import time

In [9]:
# -----------------------------
# Load and preprocess MNIST
# -----------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

In [10]:
# Normalize images
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

In [11]:
# Add channel dimension
x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]

In [12]:
# -----------------------------
# Create CNN Model
# -----------------------------
def create_model():

    model = tf.keras.Sequential([

        tf.keras.layers.Conv2D(
            filters=32,
            kernel_size=(3,3),
            activation="relu",
            input_shape=(28,28,1)
        ),

        tf.keras.layers.MaxPooling2D((2,2)),

        tf.keras.layers.Conv2D(
            filters=64,
            kernel_size=(3,3),
            activation="relu"
        ),

        tf.keras.layers.MaxPooling2D((2,2)),

        tf.keras.layers.Conv2D(
            filters=128,
            kernel_size=(3,3),
            activation="relu"
        ),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(
            128,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(
            10,
            activation="softmax"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [13]:
# -----------------------------
# Train on CPU
# -----------------------------
print("="*50)
print("Training on CPU")
print("="*50)

with tf.device("/CPU:0"):

    cpu_model = create_model()

    start = time.time()

    cpu_history = cpu_model.fit(
        x_train,
        y_train,
        epochs=5,
        batch_size=128,
        validation_split=0.1,
        verbose=1
    )

    cpu_time = time.time() - start

print(f"\nCPU Training Time: {cpu_time:.2f} seconds")

Training on CPU


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 70s 159ms/step - accuracy: 0.9204 - loss: 0.2618 - val_accuracy: 0.9797 - val_loss: 0.0658
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 78s 154ms/step - accuracy: 0.9786 - loss: 0.0707 - val_accuracy: 0.9867 - val_loss: 0.0414
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 64s 152ms/step - accuracy: 0.9852 - loss: 0.0478 - val_accuracy: 0.9897 - val_loss: 0.0360
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 65s 155ms/step - accuracy: 0.9879 - loss: 0.0371 - val_accuracy: 0.9895 - val_loss: 0.0397
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 64s 153ms/step - accuracy: 0.9903 - loss: 0.0307 - val_accuracy: 0.9908 - val_loss: 0.0343

CPU Training Time: 342.12 seconds


In [14]:
# -----------------------------
# Train on GPU
# -----------------------------
if tf.config.list_physical_devices("GPU"):

    print("\n" + "="*50)
    print("Training on GPU")
    print("="*50)

    with tf.device("/GPU:0"):

        gpu_model = create_model()

        start = time.time()

        gpu_history = gpu_model.fit(
            x_train,
            y_train,
            epochs=5,
            batch_size=128,
            validation_split=0.1,
            verbose=1
        )

        gpu_time = time.time() - start

    print(f"\nGPU Training Time: {gpu_time:.2f} seconds")

    print("\nSpeedup: {:.2f}x".format(cpu_time / gpu_time))

else:
    print("\nGPU is not available on this system.")


Training on GPU
Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.9220 - loss: 0.2514 - val_accuracy: 0.9870 - val_loss: 0.0478
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9805 - loss: 0.0643 - val_accuracy: 0.9895 - val_loss: 0.0365
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9871 - loss: 0.0431 - val_accuracy: 0.9908 - val_loss: 0.0309
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9901 - loss: 0.0319 - val_accuracy: 0.9890 - val_loss: 0.0338
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9917 - loss: 0.0269 - val_accuracy: 0.9915 - val_loss: 0.0298

GPU Training Time: 19.77 seconds

Speedup: 17.31x


## Conclusion

- Successfully trained the CNN model on both CPU and GPU.
- GPU completed training faster than CPU.
- As the model becomes larger, GPU acceleration becomes more significant.
- GPUs are highly efficient for deep learning because they perform parallel matrix computations.